# MediPredict: Exploratory Data Analysis (EDA)

This notebook conducts a thorough exploratory data analysis on the **PIMA Indians Diabetes Dataset** to understand the features, clean invalid data, analyze correlations, and prepare the dataset for deep learning binary classification.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import urllib.request
import os

# Set plots style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load Dataset

In [ ]:
dataset_url = "https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv"
dataset_path = "../dataset/diabetes.csv"

if not os.path.exists(dataset_path):
    os.makedirs("../dataset", exist_ok=True)
    urllib.request.urlretrieve(dataset_url, dataset_path)

df = pd.read_csv(dataset_path)
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Dataset Information & Summary Statistics

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Data Cleaning (Handling Missing Values)

Observe that features like `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` have a minimum value of `0.0`. A zero value for these metrics is physiologically impossible for a living human. These zeros represent missing or unrecorded data.

We will replace `0` values with `NaN` and then impute them with the median value of each column.

In [ ]:
zero_columns = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print("Number of zero values per column before cleaning:")
for col in zero_columns:
    zeros = (df[col] == 0).sum()
    print(f"- {col}: {zeros} zero values ({zeros/len(df):.2%})")

In [ ]:
# Replace zeros with NaN
for col in zero_columns:
    df[col] = df[col].replace(0, np.nan)

# Impute NaNs with column medians
for col in zero_columns:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"Imputed {col} with median: {median_val:.2f}")

print("\nVerifying zero counts after imputation:")
print(df[zero_columns].isnull().sum())

## 4. Distribution of Target Variable (`Outcome`)

In [ ]:
plt.figure(figsize=(6, 5))
sns.countplot(x='Outcome', data=df, palette=['#43a047', '#e53935'])
plt.title('Distribution of Diabetic (1) vs Non-Diabetic (0) Patients')
plt.xlabel('Outcome')
plt.ylabel('Count')
plt.show()

print("Class distribution:")
print(df['Outcome'].value_counts())
print(df['Outcome'].value_counts(normalize=True))

## 5. Feature Distribution & Histograms

In [ ]:
# Plot histograms for all numerical features
features = df.columns[:-1]
fig, axes = plt.subplots(4, 2, figsize=(15, 20))
axes = axes.flatten()

for idx, feat in enumerate(features):
    sns.histplot(df, x=feat, hue='Outcome', kde=True, ax=axes[idx], palette=['#1e88e5', '#e53935'], multiple="stack")
    axes[idx].set_title(f'Distribution of {feat} by Outcome')
    axes[idx].set_xlabel(feat)
    axes[idx].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)

sns.heatmap(corr, mask=mask, cmap=cmap, vmax=.8, center=0, annot=True, fmt=".2f",
            square=True, linewidths=.5, cbar_kws={"shrink": .5})
plt.title('Correlation Matrix of Features', fontsize=16, pad=15)
plt.show()

## 7. Conclusions & Insights
- **Glucose Levels & BMI**: Are highly correlated with the target variable `Outcome`. Higher glucose and BMI are strong indicators of diabetes susceptibility.
- **Age & Pregnancies**: Show moderate correlation, which makes sense as older individuals and those with more pregnancies might show higher instances of diabetes in this cohort.
- **Data Scaling**: Since the features have different ranges (e.g. `Insulin` vs `DiabetesPedigreeFunction`), standardizing using standard scaler will be critical for neural network training stability.